# Summerize web-based and pdf research papers
This notebook attempts at scraping research papers from the web and summerizes them
I will use closed source chatgpt and open source ollam to create an agent that takes a research paper and summerizes it.
Trafilatura is going to be used instead of BeautifulSoup because it is specifically designed for academic and article content extraction.
Why Trafilatura:
a. It had academic-specific capabilities like:
- Understands academic article structure
- Preserves citations and references
- Maintains section headers
- Handles footnotes

b. It has built in features like:
- Automatically identifies main content
- Extracts metadata (title, authors, date)
- Preserves tables and formatting
- Handles different character encodings

However trafilatura cannot handle pdf format. Also, for sites with strict access controls For very specific formatting requirements

In [4]:
import sys
#!{sys.executable} -m pip install pdfplumber

In [5]:
#imports
import os, io
import requests
#from langchain_community.document_loaders import PyPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv.main import load_dotenv
from openai import OpenAI
#from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import trafilatura #library designed specifically for scraping academic content, better than beautifulsoup
import pdfplumber
from bs4 import BeautifulSoup

import ollama



In [ ]:
#load the api key in the .env file
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

#Check the key
if not api_key:
    print("No api key found, recreate the env file")
elif not api_key.startswith("sk-proj-"):
    print("An api key found but it doesn't start with sk-proj")
elif api_key.strip() !=api_key:
    print("An api_key was found but it has extra spaces, remove them and resave the .env file")
else:
    print("Api_key found")

In [ ]:
openai = OpenAI()

message = "Hello chatgpt, I am a new user and I will require your help today"

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages = [{"role":"user","content":message}]
                 )
print(response.choices[0].message.content)

In [ ]:
response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

In [ ]:
from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')   
MODEL_llama = "llama3.2"
MODEL_deep = "deepseek-v2"

In [34]:
url = "https://datascience.codata.org/articles/10.5334/dsj-2025-007"
scraper = Paper_summerizer()
text =scraper.web_scraper(url)
#text

In [106]:
# A class that can use closed source openai or ollama
class Paper_summerizer:
    def __init__(self):
        self.header = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
        
    def summerizer_driver(self,url,ollama = False, open_ai = False):
        '''
        driver function:
        calls the scraper
            returns raw text
        calls gpt or ollama with the text scraped
            returns response with the summary
            and displays it in markdown
        '''
        #call scaraper
        self.url = url
        text = self.web_scraper(url)

        #call ollama or gpt
        if ollama: 
            summary = self.call_ollama(text) #calls call_ollama function
            return self.display_summery(summary)
        elif open_ai:
            summary = self.call_openai(text) #calls call_gpt function
            return self.display_summery(summary)
           
        else:
            print('You need to choose ollama or gpt by putting model=True')
        
            
    def web_scraper(self, url) -> dict:
        #parses the website returning a dict with text and word count
        #self.url = url
        if url.endswith('.pdf'):
            try:
                response = requests.get(url)
                pdf_file = io.BytesIO(response.content)#creates a binary stream in memory and treats it like a file object
                with pdfplumber.open(pdf_file) as pdf:
                    '''
                    pdfplumber a library that extracts text and data from pdfs. it preserves formatting,
                    handles tables, extract images, etc. It is good for academic papers which have complex formatting
                    '''
                    text = '\n'.join(page.extract_text() for page in pdf.pages)
                    return text 
            except Exception as e:
                print(f'Could not download article error {e}')
            
        else:
            try:
                download_url = trafilatura.fetch_url(url)
                if download_url:
                    text = trafilatura.extract(download_url, include_tables = True)
                    #print("text downloaded successfully")
                    return text
                             #"word_count":len(text.split())
            except Exception as e:
                print(f'Could not download article error {e}')
#---------------------
    def call_ollama(self,text):   
           #calls llm_message and returns a summary
           MODEL = 'llama3.2'
           try:
                response = ollama.chat(model=MODEL, messages= self.llm_message(text))
                return response['message']['content']
           except Exception as e:
                #if there is an error in the response then return an error message
                return f'There is an error in gpt response {e}' 
           
    def call_openai(self,text):
            #creates an openai obj
            #calls llm_message for messages and returns a summary
            try:
                openai = self.load_api_key_gpt() #calls the function that loads the api key
                
                #generate a response from gpt using gpt_message function
                response = openai.chat.completions.create(
                model = "gpt-4o-mini",
                messages = self.llm_message(text))
                return response.choices[0].message.content
            except Exception as e:
                #if there is an error in the response then return an error message
                return f'There is an error in gpt response {e}' 
#--------------------   
    def load_api_key_gpt(self):
        #load the api key in the .env file
            load_dotenv(override=True)
            api_key = os.getenv("OPENAI_API_KEY")
            #Check the key is correct
            if not api_key:
                print("No api key found, recreate the env file")
            elif not api_key.startswith("sk-proj-"):
                print("An api key found but it doesn't start with sk-proj")
            elif api_key.strip() !=api_key:
                print("An api_key was found but it has extra spaces, remove them and resave the .env file")
            else:
                print("Api_key found")
            openai = OpenAI()
            return openai
            
    def sys_prompt_gen(self):
         #function that generates a system prompt to summarize research papers
        system_prompt = system_prompt = """
                you are a research assistant focused on creating clear, structured summaries of research papers. \
                research papaer. The summury should not be more than 1000 words and it should be organized in the following sections:\
                1. SUBJECT & OBJECTIVES that includes: 
                - Primary research topic and field of study
                - Key research questions or hypotheses
                - Theoretical framework or background
                
                2. KEY FINDINGS
                - Main experimental/research results
                - Statistical significance where applicable
                - Important data points and trends
                - Supporting evidence for conclusions
                
                3. RESEARCH CHALLENGES: problems like data collection issues and sample size challenges
                
                4. CONCLUSIONS: 
                - Key findings and how these findings relate to the initial research question
                - Real-world applications or impact
                
                5. FUTURE DIRECTIONS
                The summary should include important tables
                """
        return system_prompt

     
    def user_prompt_gen(self,text):
        #function that generates a user prompt that summarizes papers
        user_prompt = "You are looking at a reseach paper, please give a detailed summary it in 1000 words or less. \
            Focus on: subject and objective, key findings, research challenges, conclusion\
            The output should be in markdown\
            If there are tables and graphs, highlight the most important ones and their key insights and it\
            includes important tables"+text
        return user_prompt
        
    def llm_message(self,text):
        return [
            {"role":"system","content":self.sys_prompt_gen()},
            {"role":"user","content":self.user_prompt_gen(text)}
                ]
    
    def display_summery(self,summary):
        
        return display(Markdown(summary))




In [94]:
url = 'https://datascience.codata.org/articles/10.5334/dsj-2025-007'
summerizer = Paper_summerizer()
summary = summerizer.summerizer_driver(url,ollama=True)
summary


The article presents the development and launch of a novel research data catalog called AMIDER (Advanced Multidisciplinary Integrated Research Database). The purpose of AMIDER is to address the need for a multidisciplinary platform that combines various scientific disciplines, making research data more open and accessible.

Key Features and Functionality:

1. **Multidisciplinary Database**: AMIDER integrates data from various scientific disciplines, creating a comprehensive database.
2. **User-Friendly Web Application**: The catalog view features thumbnails and snippets to provide an overview of the diverse datasets, while individual dataset pages offer advanced functionality such as data download and visualized data.
3. **Related Datasets Functionality**: This feature proposes relationships between datasets, enabling users to explore related data more easily.
4. **ASCII Conversion Function**: AMIDER can convert binary data formats into ASCII format for easier use.

Target Users:

1. **Non-Expert Users**: Researchers, educators, and students interested in multidisciplinary research.
2. **Individual Specialists**: Scientists and researchers working on specific projects or disciplines.

Benefits:

1. **Promoting Research Data Sharing**: AMIDER aims to encourage data sharing among scientists, fostering a culture of collaboration and open science.
2. **Enhancing Multidisciplinary Research**: By integrating diverse datasets, AMIDER facilitates research across multiple fields.
3. **Improving Data Accessibility**: The user-friendly interface and ASCII conversion function make it easier for users to access and analyze the data.

Challenges:

1. **Achieving User-Friendly Functions in a Multidisciplinary Database**: Bridging the gap between data providers (scientists) and non-expert users.
2. **Developing Advanced Functionality**: Leveraging text mining techniques for data visualization and interoperation with outside databases via metadata.

Future Prospects:

1. **Text Mining and Natural Language Processing (NLP)**: Applying NLP to create a metadata creation tool or enhance interoperation with outside databases.
2. **Collaborations and Partnerships**: Working with the Research Data Cloud project of NII, Japan, and researchers from JSPS KAKENHI.

The AMIDER system demonstrates innovative approaches to research data cataloging, aiming to promote open science, multidisciplinary research, and data accessibility. Its unique features and user-friendly interface make it an attractive platform for researchers and students alike.

In [107]:
url = 'https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf'
summerizer = Paper_summerizer()
summary = summerizer.summerizer_driver(url,open_ai=True)
summary


Api_key found


# Summary of "Attention Is All You Need"

## SUBJECT & OBJECTIVES

- **Primary Research Topic and Field of Study:**
  This paper introduces the Transformer model in the domain of sequence transduction, with a particular application to machine translation. The model's significant innovation is its reliance solely on attention mechanisms, eliminating the need for complex recurrent or convolutional neural networks.

- **Key Research Questions or Hypotheses:**
  - Can a model based solely on attention mechanisms outperform existing models that use recurrent or convolutional structures?
  - How will the proposed architecture affect training time and computational efficiency?
  - What is the impact on translation quality when using self-attention compared to traditional models?

- **Theoretical Framework or Background:**
  Existing state-of-the-art models in machine translation, such as recurrent neural networks (RNNs) and convolutional neural networks (CNNs), often struggle with parallelization and long-distance dependencies in sequences. The Transformer model is proposed to address these limitations by using self-attention, allowing it to capture global dependencies without reliance on sequential processing.

## KEY FINDINGS

- **Main Experimental/Research Results:**
  The Transformer model was tested on two datasets:
  - WMT 2014 English-to-German translation task achieved a BLEU score of 28.4.
  - WMT 2014 English-to-French translation task resulted in a BLEU score of 41.0.
  
  Both scores represent state-of-the-art performance, surpassing previous models, including ensemble methods.

- **Statistical Significance:**
  The improvements were statistically significant, leading to an increase of over 2.0 BLEU points compared to the previous best results for English-to-German translation tasks.

- **Important Data Points and Trends:**
  The Transformer can be trained significantly faster than prior architectures, achieving state-of-the-art results with training times as short as 3.5 days on eight GPUs.

- **Supporting Evidence for Conclusions:**
  The paper includes extensive empirical evidence showing the performance of the Transformer relative to previous models (Table 2), demonstrating improved translation quality and reduced training costs.

### Table 1: Model Performance Comparison
| Model | EN-DE BLEU | EN-FR BLEU | Training Cost (FLOPs) |
|-------|------------|------------|-----------------------|
| Transformer (base) | 27.3       | 38.1       | 3.3 * 10^18          |
| Transformer (big)  | 28.4       | 41.0       | 2.3 * 10^19          |
| Other models (various) | 23.75-26.36 | 39.2-41.29 | 1.0 * 10^18 - 8.0 * 10^20 |

## RESEARCH CHALLENGES

- **Problems like Data Collection Issues:** 
  The datasets utilized, including the WMT 2014 English-German and English-French datasets, are standard in the research area, reducing concerns over dataset quality but highlighting challenges in aligning and processing substantial volumes of multilingual training data.

- **Sample Size Challenges:**
  The transformer model’s reliance on parallelization allows it to work effectively with larger batches of data, although practical challenges such as the computational cost associated with training large models on significant datasets remain.

## CONCLUSIONS

- **Key Findings Related to Initial Research Question:**
  The research supports the hypothesis that a model based entirely on attention mechanisms can outperform architectures relying on sequential processing such as RNNs and CNNs. The Transformer model not only speeds up training significantly but also achieves superior translation quality.

- **Real-World Applications or Impact:**
  The results indicate that the Transformer architecture can be applied beyond machine translation, such as in various natural language processing tasks, indicating a paradigm shift in how sequence modeling tasks are approached.

## FUTURE DIRECTIONS

The authors express plans to extend their research in several areas:
- Applying attention-based models to other tasks beyond text, potentially including images and audio.
- Investigating local, restricted attention mechanisms to optimize processing large inputs and outputs.
- Developing mechanisms to reduce the sequential nature of generation for better performance on tasks requiring more immediate outputs.

The potentially transformative nature of the research reiterates the importance of effective attention mechanisms in deep learning frameworks and calls for further exploration of this architecture across varied domains.